In [4]:
import polars as pl

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os
source_bucket = os.getenv("S3_BUCKET_NAME")

In [7]:
df = pl.read_parquet(f"s3://{source_bucket}/metadata_table/metadata_20261110-223144.parquet")

In [8]:
df.head()

document_id,uuid,pdf_name,s3_key,title,subtitle,authors,editors,translators,publication_year,publisher,publication_place,edition,document_type,venue,volume,issue_number
str,str,str,str,str,str,list[str],list[str],list[str],i64,str,str,str,str,str,str,str
"""doc_76e4d42c-374""","""76e4d42c-3745-4353-aaa1-2b055c…",""" La Viena de Wittgenstein.pdf""","""google-drive-files/ La Viena d…","""La Viena de Wittgenstein""",null,"[""Allan Janik"", ""Stephen Toulmin""]",[],"[""Ignacio Gómez de Liaño""]",1998,"""Grupo Santillana de Ediciones,…","""Madrid""",null,"""Book""",null,null,null
"""doc_c25d5b87-146""","""c25d5b87-1465-4d67-9eea-43e1dc…","""((Philosophy and Social Critic…","""google-drive-files/((Philosoph…","""Foucault contra Habermas""","""Recasting the Dialogue between…",[],"[""Samantha Ashenden"", ""David Owen""]",[],1999,"""SAGE Publications""","""London • Thousand Oaks • New D…",null,"""Book""",null,null,null
"""doc_9248fae0-c8c""","""9248fae0-c8c9-4e2e-90a3-710591…","""(A History of Philosophy witho…","""google-drive-files/(A History …","""PHILOSOPHY IN THE ISLAMIC WORL…","""a history of philosophy withou…","[""PETER ADAMSON""]",[],[],2016,"""OXFORD UNIVERSITY PRESS""","""Great Clarendon Street, Oxford…","""First Edition""","""Book""",null,"""3""",null
"""doc_ea91df41-37e""","""ea91df41-37e2-4269-a9ad-919c76…","""(African Systems of Thought) P…","""google-drive-files/(African Sy…","""African Philosophy""","""Myth and Reality""","[""Paulin J. Hountondji""]","[""Charles S. Bird"", ""Ivan Karp""]","[""Henri Evans""]",1996,"""Indiana University Press""","""Bloomington and Indianapolis""","""Second Edition""","""Book""",null,null,null
"""doc_d1d2a058-3db""","""d1d2a058-3db9-4124-8fe3-2b574c…","""(Anamnesis) Graham Harman-Prin…","""google-drive-files/(Anamnesis)…","""Prince of Networks""","""Bruno Latour and Metaphysics""","[""Graham Harman""]",[],[],2009,"""re.press""","""Melbourne""",null,"""Book""",null,null,null


In [9]:
df.count()

document_id,uuid,pdf_name,s3_key,title,subtitle,authors,editors,translators,publication_year,publisher,publication_place,edition,document_type,venue,volume,issue_number
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
5280,5280,5280,5280,5061,1969,5280,5280,5280,3668,3604,2651,1005,4049,1060,922,498


In [10]:
# Check null
df.select(pl.col("title").is_null().sum())

title
u32
219


Fields to add:
- isbn (for books)
- DOI (for journal articles)

## Inspect CrossRef API

In [12]:
import requests
import json

title = "Foucault contra Habermas"

url = "https://api.crossref.org/works"

params = {
    "query.title": title,
    "rows": 5,
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

items = data["message"]["items"]

first_result = items[0] if items else None

In [13]:
first_result

{'indexed': {'date-parts': [[2024, 8, 8]],
  'date-time': '2024-08-08T04:04:16Z',
  'timestamp': 1723089856792},
 'reference-count': 0,
 'publisher': 'Philosophy Documentation Center',
 'issue': '1',
 'content-domain': {'domain': [], 'crossmark-restriction': False},
 'DOI': '10.5840/philtoday200751154',
 'type': 'journal-article',
 'created': {'date-parts': [[2015, 10, 10]],
  'date-time': '2015-10-10T05:02:50Z',
  'timestamp': 1444453370000},
 'page': '3-16',
 'source': 'Crossref',
 'is-referenced-by-count': 0,
 'title': ['Foucault Contra Habermas'],
 'prefix': '10.5840',
 'volume': '51',
 'author': [{'given': 'Day',
   'family': 'Wong',
   'sequence': 'first',
   'affiliation': [],
   'role': [{'role': 'author', 'vocabulary': 'crossref'}]},
  {'name': 'DePaul University',
   'sequence': 'additional',
   'affiliation': [],
   'role': [{'role': 'author', 'vocabulary': 'crossref'}]}],
 'member': '3884',
 'published-online': {'date-parts': [[2007]]},
 'container-title': ['Philosophy Toda

In [ ]:
def extract_crossref_metadata(work):
    # Extract publication year
    date = (
        work.get("published-print")
        or work.get("published-online")
        or work.get("published")
        or work.get("issued")
    )

    publication_year = (
        date["date-parts"][0][0]
        if date and date.get("date-parts")
        else None
    )

    # Extract authors, editors, and translators
    contributors = work.get("author", []) + work.get("editor", []) + work.get("translator", [])

    def get_names(role):
        return [
            (
                person.get("name")
                or " ".join(filter(None, [
                    person.get("given"),
                    person.get("family")
                ]))
            )
            for person in work.get(role, [])
        ]

    return {
        "title": (work.get("title") or [None])[0],
        "subtitle": (work.get("subtitle") or [None])[0],
        "authors": get_names("author"),
        "editors": get_names("editor"),
        "translators": get_names("translator"),
        "publication_year": publication_year,
        "publisher": work.get("publisher"),
        "publication_place": work.get("publisher-location"),
        "edition": work.get("edition-number"),
        "document_type": work.get("type"),
        "venue": (work.get("container-title") or [None])[0],
        "volume": work.get("volume"),
        "issue_number": work.get("issue"),
        "ISBN": work.get("ISBN"),
        "DOI": work.get("DOI"),
        "ISSN": work.get("ISSN"),
    }